In [54]:
from dataAnalysis.DataAnalysis import DataAnalysis
import pandas as pd
from torch.nn.functional import normalize
from sklearn.ensemble import RandomForestClassifier
from EnsembleFramework import Framework
from sklearn.metrics import roc_curve
from numpy import sqrt, argmax
from joblib import dump, load
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
import shap

In [55]:
data = pd.read_csv(r"extdata/sbcdata.csv", header=0)
data_analysis = DataAnalysis(data)


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [56]:
sorted_train_data = data_analysis.get_training_data().sort_values(by="Id").reset_index(drop = True)
train_df = sorted_train_data.loc[:sorted_train_data.shape[0]*.8,:]
val_df = sorted_train_data.loc[sorted_train_data.shape[0]*.8:,:]
test_df = data_analysis.get_testing_data()
gw_df = data_analysis.get_gw_testing_data()

In [33]:
# test_df = pd.read_csv("leipzig.csv", sep = ";")
# test_df["Label"] = test_df["Label"].replace({1: "Sepsis", 0: "Control"})

# gw_df = pd.read_csv("gw.csv", sep = ";")
# gw_df["Label"] = gw_df["Label"].replace({1: "Sepsis", 0: "Control"})

In [57]:
import numpy as np
import torch
from dataAnalysis.Constants import FEATURES, LABEL_COLUMN_NAME

def convert_cat_feature(df):
    df = df.copy()
    df["SexCategory"] = (df["Sex"] == "W").astype(int)
    return df
    
def get_graph(df):
    edge_index = []
    df = convert_cat_feature(df)
    df = df.sort_values(by=["Id", "Time"]).reset_index(drop=True)
    
    ## group df by ids
    for identifier, group in df.groupby("Id"):
        offset = group.index[0]
        triu_matrix = np.triu((group.index.values + np.identity(1))[0])
        triu_exp_matrix = np.expand_dims(triu_matrix, axis=-1)
    
        idx_shape = group.index.shape[0]
        idx_matrix = np.ones((idx_shape, idx_shape)) * np.arange(idx_shape) + 1 + offset
        idx_matrix = np.transpose(idx_matrix)
        idx_exp_matrix = np.expand_dims(idx_matrix, axis=-1)
    
        unprocess_edges = np.concatenate((idx_exp_matrix, triu_exp_matrix), axis=-1)
        reshaped_unprocess_edges = np.reshape(unprocess_edges, (-1, 2))
        mask = (reshaped_unprocess_edges[:, 0] * reshaped_unprocess_edges[:, 1]) != 0
        edge_index.append((reshaped_unprocess_edges[mask] - 1).astype(np.int64))
    edge_index_torch = torch.from_numpy(np.concatenate(edge_index)).type(torch.long).transpose(0,1)
    features_torch = torch.from_numpy(df[FEATURES].values).type(torch.float)
    labels_torch = torch.from_numpy((df.sort_values(by=["Id", "Time"])[LABEL_COLUMN_NAME] == "Sepsis").values).type(torch.long)
    return features_torch, edge_index_torch, labels_torch

In [58]:
X_train_comp, edge_index_train_comp, y_train_comp = get_graph(sorted_train_data)
X_train, edge_index_train, y_train = get_graph(train_df)
X_val, edge_index_val, y_val = get_graph(val_df)
X_test, edge_index_test, y_test = get_graph(test_df)
X_gw, edge_index_gw, y_gw = get_graph(gw_df)

In [59]:
rev_edge_index_train_comp = torch.zeros_like(edge_index_train_comp)
rev_edge_index_train_comp[0,:] = edge_index_train_comp[1,:]
rev_edge_index_train_comp[1,:] = edge_index_train_comp[0,:]

rev_edge_index_train = torch.zeros_like(edge_index_train)
rev_edge_index_train[0,:] = edge_index_train[1,:]
rev_edge_index_train[1,:] = edge_index_train[0,:]

rev_edge_index_val = torch.zeros_like(edge_index_val)
rev_edge_index_val[0,:] = edge_index_val[1,:]
rev_edge_index_val[1,:] = edge_index_val[0,:]

rev_edge_index_test = torch.zeros_like(edge_index_test)
rev_edge_index_test[0,:] = edge_index_test[1,:]
rev_edge_index_test[1,:] = edge_index_test[0,:]

rev_edge_index_gw = torch.zeros_like(edge_index_gw)
rev_edge_index_gw[0,:] = edge_index_gw[1,:]
rev_edge_index_gw[1,:] = edge_index_gw[0,:]

In [60]:
dir_sets = [("train_comp", X_train_comp, edge_index_train_comp, y_train_comp), ("train", X_train, edge_index_train, y_train), ("val", X_val, edge_index_val, y_val), ("test", X_test, edge_index_test, y_test),
       ("gw", X_gw, edge_index_gw, y_gw)]
dir_sets_dict = {dir_set[0]: (dir_set[1:]) for dir_set in dir_sets}
rev_dir_sets = [("train_comp", X_train_comp, rev_edge_index_train_comp, y_train_comp), ("train", X_train, rev_edge_index_train, y_train), ("val", X_val, rev_edge_index_val, y_val), ("test", X_test, rev_edge_index_test, y_test),
       ("gw", X_gw, rev_edge_index_gw, y_gw)]
rev_dir_sets_dict = {rev_dir_set[0]: (rev_dir_set[1:]) for rev_dir_set in rev_dir_sets}

In [61]:
def get_features(framework, X, edge_index):
    return framework.get_features(X, edge_index, torch.ones(X.shape[0]).type(torch.bool))

In [62]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
def test_auroc_and_auprc(framework, clf, X, edge_index,y):
    features = torch.cat(get_features(framework, X, edge_index), dim = 1)
    pred_proba = clf.predict_proba(features.cpu())[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    return auroc, auprc

In [63]:
edge_type_sets = {
    "dir": dir_sets_dict,
    "rev_dir": rev_dir_sets_dict,
    # "undir": undir_sets_dict,
}

In [66]:
from joblib import load

rf = load("prospective_random_forest.joblib")

In [67]:
rf_rev = load("retrospective_random_forest.joblib")

In [68]:
def user_function(kwargs):
    return kwargs["updated_features"] - kwargs["mean_neighbors"]

hops = [0, 1]
framework = Framework(user_functions=[user_function, user_function], 
                             hops_list=[0,1],
                             clfs=[None, None],
                             gpu_idx=0,
                             handle_nan=0.0,
                            attention_configs=[ None, None])

In [69]:
X, edge_index,y = edge_type_sets["rev_dir"]["test"]
test_auroc_and_auprc(framework, rf_rev, X, edge_index, y)

(0.941869275034136, 0.043792621380179635)

In [52]:
test_df.columns

Index(['Id', 'Time', 'Age', 'Sex', 'HGB', 'WBC', 'RBC', 'MCV', 'PLT', 'Label'], dtype='object')

In [72]:
features = torch.cat(get_features(framework, X, edge_index), dim = 1)
pred_proba = rf_rev.predict_proba(features.cpu())[:,1]

In [78]:
features[10, :]

tensor([ 61.0000,   0.0000,  10.6000,   7.5000,   5.4600,  91.4000, 282.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000],
       device='cuda:0')

In [79]:
pred_proba[10]

0.20411742550749656

In [81]:
exp = shap.TreeExplainer(rf_rev)

In [86]:
exp.shap_values(features[10, :].cpu().numpy())[-1][:7]

array([ 0.00489133,  0.00242337, -0.01928404, -0.2003469 , -0.02306462,
       -0.00069235, -0.06590538])

In [87]:
exp.shap_values(features[10, :].cpu().numpy())[-1][7:]

array([2.26430868e-05, 0.00000000e+00, 2.29457773e-02, 1.92879862e-02,
       2.62878029e-02, 1.52600108e-02, 1.59379845e-02])

In [88]:
exp.shap_values(features[10, :].cpu().numpy())[-1][7:] + exp.shap_values(features[10, :].cpu().numpy())[-1][:7]

array([ 0.00491398,  0.00242337,  0.00366174, -0.18105891,  0.00322318,
        0.01456766, -0.04996739])